# Calculate sectors footprint with Exiobase

**Related publication**

 - Title: Residual biomass to bio-based chemicals and plastics: ex-ante screening methodology for prioritizing high-impact substitutions
 - Authors: [Nicolas LIENART](https://orcid.org/0009-0001-3259-2819), [Thibaut LECOMPTE](https://orcid.org/0000-0001-9237-8454), [Lorie HAMELIN](https://orcid.org/0000-0001-9092-1900) 
 - Journal: Resources, Conservation and Recycling (RCR) - Elsevier
 - Doi: #todo
 - Git repository (Forge INRAE): https://forge.inrae.fr/nicolas.lienart/screen-lca-paper-supplementary-code
 - Git repository (GitHub): https://github.com/nicolnt/screen-lca-paper-supplementary-code

**Description and details**

This code calculates the greenhouse gas (GHG) footprint of France (FR) imports and production for year 2019 based on Exiobase data. The GHG indicator is the GWP100 calculated using coefficients from IPCC AR6 2021. Refer to the aforementioned main manuscript and accompanying supplementary information documents for more details.

**Updates**

 - May 04, 2026: Ready for submission
 - August 08, 2026: Updated variable names and clarified some functions. Reorganized the heading structure for consistency.

**Package versions**

 - See [`environment.yml`](./environment.yml)
 - pymrio library code fetched directly from GitHub. See: https://github.com/IndEcol/pymrio/issues/144

 - Exiobase version: 3.9.5
   - dataset: IOT_2019_pxp
   - Available at https://doi.org/10.5281/zenodo.14869924

**Relevant references**

 - Pellan, M. et al. (2024) “Integrating Consumption-Based Metrics into Sectoral Carbon Budgets to Enhance Sustainability Monitoring of Building Activities,” Sustainability, 16(16), p. 6762. Available at: https://doi.org/10.3390/su16166762.
 <br>_Provided insiration on the use of Pymrio with Exiobase, see associated Github repository_
 - Aguilar-Hernandez, G.A. (2025) “A Novel Framework to Measure Circularity Trade-offs and Synergies in the Global Context,” Journal of Circular Economy, 3(1). Available at: https://doi.org/10.55845/YTGD9041.
 <br>_Provided the Python code for GHG stressors characterization, see associated Github repository_
 - Andrieu, B. et al. (2024) “An open-access web application to visualise countries’ and regions’ carbon footprints using Sankey diagrams,” Communications Earth & Environment, 5(1), pp. 1–9. Available at: https://doi.org/10.1038/s43247-024-01378-8.
  <br>_Provided insiration on the use of Pymrio with Exiobase, see associated Github repository_
 - Wiedmann, T. (2017) “On the decomposition of total impact multipliers in a supply and use framework,” Journal of Economic Structures, 6(1), p. 11. Available at: https://doi.org/10.1186/s40008-017-0072-0.
  <br>_Provided example of input-output calculations and results_
 - Stadler, K. (2021) “Pymrio – A Python Based Multi-Regional Input-Output Analysis Toolbox,” Journal of Open Research Software, 9(1). Available at: https://doi.org/10.5334/jors.251.
 <br>_Provided example of input-output calculations and results_
 - https://pymrio.readthedocs.io/en/latest/
 <br>_Pymrio library documentation_

## Initialization

In [1]:
import pymrio
import numpy as np
import pandas as pd
from datetime import datetime

In [2]:
print('numpy version:', np.__version__)
print('pandas version:', pd.__version__)
print('pymrio version:', pymrio.__version__)

numpy version: 2.4.3
pandas version: 2.2.3
pymrio version: 0.6.3


In [3]:
OUTPUT_PATH = 'output/'
IO_DATA_PATH = './Exiobase_data/'
EXIOBASE_VERSION = 'EXIOBASE_v3.9.5'
EXIOBASE_MODEL_AND_YEAR = 'IOT_2019_pxp'
EXIOBASE_PATH = IO_DATA_PATH + EXIOBASE_VERSION + '/' + EXIOBASE_MODEL_AND_YEAR
VALUE_COLUMN_NAME = 'value (kg CO2 eq.)'

print(EXIOBASE_PATH)

./Exiobase_data/EXIOBASE_v3.9.5/IOT_2019_pxp


In [4]:
# NOTE: Load Exiobase data
# Takes < 1 min
exiobase = pymrio.parse_exiobase3(EXIOBASE_PATH)

In [5]:
# NOTE: Export function (optional)
# USAGE: export_csv(exiobase, exiobase.impacts.D_cba, 'name')
def export_csv(exiobase, data, name):
    date = datetime.now().strftime('%Y%m%d')
    version = "exiobase_v" + exiobase.meta.system + exiobase.meta.description[len(exiobase.meta.description) - 4:]
    #print(date + " - " + version)
    data.to_csv(OUTPUT_PATH + date + ' - output_' + version + '_' + name + '.csv')

## Stressors characterization

Manual characterization originally inspired from https://github.com/aguilarga/circularity_trade-offs-synergies_supplementary_material/blob/main/ghg_calculation_exiobase_v3.9.5.py
Linked to this publication: Aguilar-Hernandez, G.A. (2025) “A Novel Framework to Measure Circularity Trade-offs and Synergies in the Global Context,” Journal of Circular Economy, 3(1). Available at: https://doi.org/10.55845/YTGD9041.

However, some flows where missing in this work and added.
All the cheracterization factor were taken from IPCC 2021 method, obtained with the help of Brightway.

Brightway method key: `('ecoinvent-3.11', 'IPCC 2021', 'climate change: total (excl. biogenic CO2)', 'global warming potential (GWP100)')`

Now following the official `pymrio` method to characterize stressors: https://pymrio.readthedocs.io/en/latest/notebooks/stressor_characterization.html

In [6]:
gwp_characterization_table = pd.read_csv('IPCC 2021 GWP100 characterization.csv')

In [7]:
characterized_air_emissions = exiobase.air_emissions.characterize(gwp_characterization_table)

In [8]:
exiobase.ghg_impacts = characterized_air_emissions.extension

In [18]:
# NOTE: Calculate all matrices (incl. L, S, D_cba, D_pba), including those related to the characterized impacts

# Can take a few minutes
# exiobase.calc_all()

# Much faster (< 1 min)
exiobase.ghg_impacts.calc_system(exiobase.x, exiobase.Y, L=pymrio.calc_L(exiobase.A))

## Accounting approach: Footprint of production + imports (as used in the main paper case study)

### 1.1 Imported final demand footprint in France per product

In [20]:
Y_FR_imports = exiobase.Y.loc[:, ['FR']]

Y_FR_imports.loc[['FR'], ['FR']] = 0

In [21]:
Y_FR_imports = exiobase.Y.loc[:, ['FR']]
#specific_index_columns

# NOTE: Setting FR final demand satisfied by FR to 0. Avoiding double counting with FR production footprint
Y_FR_imports.loc[['FR'], ['FR']] = 0

# NOTE: Join the different final demand category columns together, no distinction is made
Y_FR_imports = Y_FR_imports.sum(1)

diag_Y_FR_imports = pd.DataFrame(np.diag(Y_FR_imports))
diag_Y_FR_imports.index = Y_FR_imports.index 
diag_Y_FR_imports.columns = Y_FR_imports.index

In [22]:
# NOTE: Multiply FR final demand with the footprint data
Y_FR_imports_footprint = exiobase.ghg_impacts.M.dot(diag_Y_FR_imports)
# Y_FR_imports_footprint

In [26]:
Y_FR_imports_footprint_agg = Y_FR_imports_footprint.T.groupby('sector').sum()
Y_FR_imports_footprint_agg

impact,climate change: total (excl. biogenic CO2)
sector,
Additives/Blending Components,5.830062e-04
Air transport services (62),2.012679e+10
Aluminium and aluminium products,2.051720e+07
Aluminium ores and concentrates,7.233621e+04
Animal products nec,2.632887e+08
...,...
"Wood material for treatment, Re-processing of secondary wood material into new wood material",0.000000e+00
Wood waste for treatment: incineration,0.000000e+00
Wood waste for treatment: landfill,0.000000e+00


In [27]:
# NOTE: Export restults to CSV
export_csv(exiobase, Y_FR_imports_footprint_agg, "Y_FR_imports_footprint")

### 1.2 Imported intermediate demand footprint in France per product

In [28]:
# NOTE: Only conserve intermediate industry demand from FR
Z_FR_imports = exiobase.Z.loc[:, ['FR']]

# NOTE: Sum all intermediate demand categories (columns) together,
# no distinction is made regarding the origin of the industry
Z_FR_imports = Z_FR_imports.sum(1)

# NOTE: Set locally satisfied intermediate demand from FR to 0.
# Avoiding double counting with FR production footprint
Z_FR_imports.loc['FR'] = 0

diag_Z_FR_imports = pd.DataFrame(np.diag(Z_FR_imports))
diag_Z_FR_imports.index = Z_FR_imports.index 
diag_Z_FR_imports.columns = Z_FR_imports.index
#diag_Z_FR

In [29]:
# NOTE: Multiply FR intermediate demand with the footprint data

# For EXIOBASE v3.9.5
Z_FR_imports_footprint = exiobase.ghg_impacts.M.dot(diag_Z_FR_imports)

# For EXIOBASE v3.8.2
# Z_FR_imports_footprint = exiobase.impacts.M.dot(diag_Z_FR_imports)

In [30]:
Z_FR_imports_footprint_agg = Z_FR_imports_footprint.T.groupby('sector').sum()
Z_FR_imports_footprint_agg

impact,climate change: total (excl. biogenic CO2)
sector,
Additives/Blending Components,1.662453e+08
Air transport services (62),3.147313e+09
Aluminium and aluminium products,5.854981e+09
Aluminium ores and concentrates,7.682940e+07
Animal products nec,1.360654e+07
...,...
"Wood material for treatment, Re-processing of secondary wood material into new wood material",0.000000e+00
Wood waste for treatment: incineration,6.243292e+06
Wood waste for treatment: landfill,3.516346e+06


In [31]:
export_csv(exiobase, Z_FR_imports_footprint_agg, "Z_FR_imports_footprint")

### 2.1 Production footprint in France per product, for final demand (FR and exports)

In [32]:
Y_FR_production = exiobase.Y.copy()

# NOTE: all other producers in Z than FR are set to 0, we only consider production happening in FR
Y_FR_production.loc[Y_FR_production.index.get_level_values('region') != 'FR'] = 0

Y_FR_production

# NOTE: Join the different final demand category columns together, no distinction is made
Y_FR_production = Y_FR_production.sum(1)


diag_Y_FR_production = pd.DataFrame(np.diag(Y_FR_production))
diag_Y_FR_production.index = Y_FR_production.index 
diag_Y_FR_production.columns = Y_FR_production.index

In [33]:
# NOTE: Multiply FR production feeding different final demand with the footprint data

# For EXIOBASE v3.9.5
Y_FR_production_footprint = exiobase.ghg_impacts.M.dot(diag_Y_FR_production)

# For EXIOBASE v3.8.2
# Y_FR_production_footprint exiobase.impacts.M.dot(diag_Y_FR_production)

In [34]:
Y_FR_production_footprint_agg = Y_FR_production_footprint.T.groupby('sector').sum()
Y_FR_production_footprint_agg

impact,climate change: total (excl. biogenic CO2)
sector,
Additives/Blending Components,5.261586e+05
Air transport services (62),1.659652e+10
Aluminium and aluminium products,8.576190e+07
Aluminium ores and concentrates,5.638289e+04
Animal products nec,1.603625e+09
...,...
"Wood material for treatment, Re-processing of secondary wood material into new wood material",0.000000e+00
Wood waste for treatment: incineration,3.283354e+07
Wood waste for treatment: landfill,3.692305e+07


In [35]:
export_csv(exiobase, Y_FR_production_footprint_agg, "Y_FR_production_footprint")

### 2.2 Production footprint in France per product, for intermediate demand (FR and exports)

In [36]:
Z_FR_agg = exiobase.Z.copy()

# NOTE: all other producers in Z than FR are set to 0, we only consider production happening in FR
Z_FR_agg.loc[Z_FR_agg.index.get_level_values('region') != 'FR'] = 0


# NOTE: Only conserve industry production from FR.
# Dimensions: 2D (product) x 2D (region x product) | 200 x 9800
# Z_FR = exiobase.Z.loc[[('FR')]]
Z_FR_agg

# NOTE: Sum all output categories (columns) together,
# No distinction is made regarding the demanding industry
Z_FR_agg = Z_FR_agg.sum(1)
Z_FR_agg

# NOTE: Diagonalized Z (gross output) where only FR output is kept
# Dimensions: 2D (region x product) x 2D (region x product) | 9800 x 9800
diag_Z_FR_agg = pd.DataFrame(np.diag(Z_FR_agg))
diag_Z_FR_agg.index = Z_FR_agg.index 
diag_Z_FR_agg.columns = Z_FR_agg.index
# diag_Z_FR_agg

In [37]:
# NOTE: Multiply FR production with the footprint data.

# For EXIOBASE v3.8.2
# Z_FR_production_footprint = exiobase.impacts.M.dot(diag_Z_FR)

# For EXIOBASE v3.9.5
Z_FR_production_footprint = exiobase.ghg_impacts.M.dot(diag_Z_FR_agg)


In [38]:
Z_FR_production_footprint_agg = Z_FR_production_footprint.T.groupby('sector').sum()
Z_FR_production_footprint_agg

impact,climate change: total (excl. biogenic CO2)
sector,
Additives/Blending Components,1.835913e+08
Air transport services (62),1.360946e+10
Aluminium and aluminium products,2.382755e+09
Aluminium ores and concentrates,1.892917e+07
Animal products nec,6.004453e+08
...,...
"Wood material for treatment, Re-processing of secondary wood material into new wood material",0.000000e+00
Wood waste for treatment: incineration,1.780554e+08
Wood waste for treatment: landfill,1.661176e+08


In [39]:
export_csv(exiobase, Z_FR_production_footprint_agg, "Z_FR_production_footprint")